# Specialist Recommendation Model

This notebook builds the specialist recommendation module for the NexusAI platform.

## Goal

Recommend the most relevant specialists for a project using:

- semantic similarity between project and specialist profile,
- skills matching,
- sector matching,
- rating,
- availability,
- budget, language and location compatibility.

## Important Product Decision

This module is **not** part of the main business validation score.

The validation score evaluates the idea.  
The specialist module helps the entrepreneur improve the idea by finding the right experts.


## Why We Use an Internal / Synthetic Dataset

A public dataset is not the best option for this feature because NexusAI must recommend the specialists registered in the platform.

In production, the data will come from MongoDB:

- `specialists`
- `projects`
- `specialist_recommendations`

During development, we use a synthetic CSV to test the recommendation logic.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from specialist_recommendation_engine import (
    DEFAULT_SPECIALIST_WEIGHTS,
    SpecialistRecommendationEngine,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_PATH = Path("data/specialists_sample.csv")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)


## 1. Load Specialist Profiles


In [ ]:
specialists_df = pd.read_csv(DATA_PATH)

print("Dataset shape:", specialists_df.shape)
print("Columns:", list(specialists_df.columns))
display(specialists_df.head())


## 2. Define Example Project

This example represents a project submitted by an entrepreneur.


In [ ]:
project_request = {
    "project_id": "p_001",
    "title": "AI business validation platform",
    "description": (
        "A SaaS platform that uses AI, sentiment analysis and market research "
        "to validate startup ideas and generate recommendations."
    ),
    "sector": "SaaS",
    "needs": ["AI", "market research", "go-to-market", "business plan"],
    "project_stage": "idea",
    "budget_per_hour": 65,
    "preferred_language": "fr",
    "location": "Morocco",
    "top_k": 5,
}

project_request


## 3. Run Hybrid Recommendation

The recommendation score is calculated as:

| Component | Weight |
|---|---:|
| Semantic similarity | 40% |
| Skills match | 25% |
| Sector match | 15% |
| Rating | 10% |
| Availability | 5% |
| Budget / language / location | 5% |


In [ ]:
engine = SpecialistRecommendationEngine()
recommendations = engine.recommend(project_request, sample_csv_path=DATA_PATH)

recommendations_df = pd.DataFrame(
    [
        {
            "specialist_id": item.specialist_id,
            "full_name": item.full_name,
            "expertise_domain": item.expertise_domain,
            "recommended_score": item.recommended_score,
            "reason": item.reason,
            **item.score_details,
        }
        for item in recommendations
    ]
)

display(recommendations_df)


## 4. Visualize Recommendation Ranking


In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(
    data=recommendations_df,
    x="recommended_score",
    y="full_name",
    hue="expertise_domain",
    dodge=False,
    palette="viridis",
)
plt.title("Top Specialist Recommendations")
plt.xlabel("Recommendation Score")
plt.ylabel("Specialist")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
score_columns = [
    "semantic_similarity",
    "skills_match",
    "sector_match",
    "rating",
    "availability",
    "budget_language_location",
]

heatmap_df = recommendations_df.set_index("full_name")[score_columns]

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_df, annot=True, cmap="YlGnBu", fmt=".1f", vmin=0, vmax=100)
plt.title("Explanation of Specialist Recommendation Scores")
plt.tight_layout()
plt.show()


## 5. Empty Database Behavior

If MongoDB has no specialists yet, the engine returns an empty list.

This is expected behavior. The platform can show an empty state:

> Aucun spécialiste disponible pour le moment.


In [ ]:
empty_result = engine.recommend(project_request, specialists=[])
print("Recommendations with empty specialist database:", empty_result)


## 6. Save Configuration Artifact

The recommendation configuration is saved for traceability and MLOps documentation.


In [ ]:
artifact_path = ARTIFACTS_DIR / "specialist_recommendation_config.joblib"

joblib.dump(
    {
        "module": "SpecialistRecommendationEngine",
        "method": "hybrid weighted score + token semantic similarity",
        "weights": DEFAULT_SPECIALIST_WEIGHTS,
        "uses_public_dataset": False,
        "production_data_source": "MongoDB specialists collection",
    },
    artifact_path,
)

print(f"Specialist recommendation config saved to: {artifact_path.resolve()}")


## 7. Conclusion

This module is ready for the MVP:

- it works with a synthetic dataset now,
- it will work with real MongoDB specialist profiles later,
- it returns explainable recommendation scores,
- it can be exposed through FastAPI for Spring Boot integration.
